# 24 | LangChain 自定义 Middleware：让 Agent 按工程规则办事

内置 Middleware 解决的是常见问题，比如限流、重试、人工审批、PII 脱敏、上下文清理。

但真实项目里，总会遇到一些更细的业务规则：

- 模型调用前要打审计日志
- 本地模型失败后先重试，再切云端模型
- 工具调用前要检查参数
- 大批量导出不能让 Agent 直接创建任务
- 模型返回后要记录它有没有发起工具调用

这些就适合用自定义 Middleware。

这篇用一个贴近生产的例子：**订单报表导出 Agent**。它要能查权限、创建导出任务，但不能让模型一激动就导出 100 万行。

## 一、这个场景为什么需要自定义 Middleware

假设用户说：

> 帮我给财务创建一个订单报表导出任务，预计 200000 行。

如果没有自定义控制，Agent 可能直接调用工具创建任务。问题是：

- 200000 行导出可能拖垮后台任务
- 本地 Ollama 服务可能临时不可用
- 工具调用有没有发生，事后不好追踪
- 模型有没有乱填参数，也很难定位

所以我们要加几层控制：

| 需求 | 用什么 hook |
|---|---|
| 模型调用前记录上下文数量 | `@before_model` |
| 模型返回后记录是否调用工具 | `@after_model` |
| 本地模型失败后重试，再切云端 | `@wrap_model_call` |
| 工具调用前审计和拦截大导出 | `@wrap_tool_call` |
| 多个规则打包复用 | `AgentMiddleware` |

自定义 Middleware 的价值就在这里：把业务规则放在 Agent 执行链路里，而不是散落在每个工具函数里。

## 二、准备模型和工具

这里让本地 Ollama `qwen2.5:14b` 做主模型。

如果本地模型连续失败，就切到云端 `deepseek-v4-flash`。这样既能优先用本地模型省成本，也不会因为本地服务临时不可用就让 Agent 当场下班。

In [ ]:
import os
import time
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentMiddleware,
    AgentState,
    after_model,
    before_model,
    wrap_model_call,
    wrap_tool_call,
)
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import ToolMessage

load_dotenv(dotenv_path=".env")

# 主模型：本地 Ollama。优先用它，成本低，也适合本地开发。
local_model = init_chat_model("ollama:qwen2.5:14b")

# 备用模型：云端模型。本地模型失败多次后切到它。
cloud_model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
)


class ExportTaskInput(BaseModel):
    """创建报表导出任务的参数结构。"""

    report_name: str = Field(description="报表名称，例如 order_export")
    file_format: Literal["xlsx", "csv"] = Field(description="导出格式")
    estimated_rows: int = Field(description="预计导出行数")
    reason: str = Field(description="创建导出任务的原因")


@tool
def check_export_permission(user_role: str, region: str) -> dict:
    """查询当前用户是否有订单报表导出权限。"""
    return {
        "user_role": user_role,
        "region": region,
        "can_export": user_role in ["finance", "ops_manager"],
        "scope": "all" if user_role == "finance" else region,
    }


@tool(args_schema=ExportTaskInput)
def create_export_task(
    report_name: str,
    file_format: Literal["xlsx", "csv"],
    estimated_rows: int,
    reason: str,
) -> dict:
    """创建订单报表导出任务。"""
    print(
        f"[导出工具] 创建任务：report={report_name}, "
        f"format={file_format}, rows={estimated_rows}, reason={reason}"
    )
    return {
        "task_id": "EXPORT-20260606-001",
        "report_name": report_name,
        "file_format": file_format,
        "estimated_rows": estimated_rows,
        "status": "queued",
    }

## 三、先用 decorator 写几个小 Middleware

Decorator 写法适合单个 hook：简单、直接、改起来快。

下面四个 Middleware 分别拦截模型调用前、模型调用后、模型调用过程、工具调用过程。

In [ ]:
@before_model
def log_before_model(state: AgentState, runtime):
    """模型调用前执行：记录本轮上下文规模。"""
    print(f"[审计] 准备调用模型，messages={len(state['messages'])}")
    return None


@after_model
def log_after_model(state: AgentState, runtime):
    """模型调用后执行：看看模型是直接回复，还是准备调用工具。"""
    last_message = state["messages"][-1]
    tool_calls = getattr(last_message, "tool_calls", [])

    if tool_calls:
        tool_names = [call["name"] for call in tool_calls]
        print(f"[审计] 模型发起工具调用：{tool_names}")
    else:
        content = str(last_message.content)
        print(f"[审计] 模型直接回复，长度={len(content)}")

    return None


@wrap_model_call
def retry_local_then_cloud(request, handler):
    """模型调用外层执行：本地模型先重试，仍失败再切云端模型。"""
    last_error = None

    for attempt in range(1, 4):
        try:
            print(f"[模型调用] 本地 qwen2.5 第 {attempt} 次尝试")
            # request.override(model=...) 会生成一个换了模型的新请求。
            return handler(request.override(model=local_model))
        except Exception as e:
            last_error = e
            print(f"[模型调用] 本地模型失败：{type(e).__name__}")

    print("[模型调用] 本地模型连续失败，切换到云端 deepseek-v4-flash")
    try:
        return handler(request.override(model=cloud_model))
    except Exception:
        print(f"[模型调用] 云端模型也失败，最后一次本地错误：{type(last_error).__name__}")
        raise


@wrap_tool_call
def audit_and_guard_tool(request, handler):
    """工具调用外层执行：先审计参数，再决定是否放行。"""
    tool_name = request.tool_call["name"]
    args = request.tool_call.get("args", {})
    print(f"[工具审计] 准备调用 {tool_name}，args={args}")

    # 业务规则：预计导出行数太大时，不让 Agent 直接创建任务。
    if tool_name == "create_export_task" and args.get("estimated_rows", 0) > 100_000:
        return ToolMessage(
            name=tool_name,
            tool_call_id=request.tool_call["id"],
            content="预计导出行数超过 100000，请先走人工审批或拆分导出范围。",
        )

    start = time.perf_counter()
    result = handler(request)  # 真正执行工具。
    elapsed_ms = (time.perf_counter() - start) * 1000
    print(f"[工具审计] {tool_name} 执行完成，耗时 {elapsed_ms:.1f}ms")
    return result

这几个 hook 的区别很清楚：

- `before_model` / `after_model`：像模型调用前后的检查点
- `wrap_model_call`：包住模型调用，可以重试、换模型、拦截错误
- `wrap_tool_call`：包住工具调用，可以审计参数、拦截危险操作、统计耗时

如果只是打印日志，普通函数也能做。但如果要把规则插进 Agent 的执行链路里，Middleware 更干净。

## 四、把 Middleware 装进 Agent

这段代码可以直接测试：

- 如果本地 Ollama 正常，会优先走 `qwen2.5:14b`
- 如果你关闭 Ollama，本地模型会失败 3 次，然后切到云端模型
- 如果模型尝试创建超过 100000 行的导出任务，工具 Middleware 会拦住

In [ ]:
export_agent = create_agent(
    model=local_model,
    tools=[check_export_permission, create_export_task],
    middleware=[
        retry_local_then_cloud,
        log_before_model,
        log_after_model,
        audit_and_guard_tool,
    ],
    system_prompt=(
        "你是订单报表导出 Agent。"
        "创建导出任务前，先检查用户权限。"
        "如果用户明确要求创建导出任务，需要调用 create_export_task。"
        "如果工具提示需要人工审批，就不要继续强行创建任务。"
    ),
)

result = export_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "我是 finance，区域是 east。"
                    "请帮我创建订单报表导出任务，格式 xlsx，预计 200000 行，"
                    "原因是月度财务结算。"
                ),
            }
        ]
    }
)

print("\n[最终回复]")
print(result["messages"][-1].content)

如果你想测试“本地失败后切云端”，可以先停掉 Ollama 服务，再执行上面这段。

这段输出可以分成三层看。

**第一层：模型调用前审计**

```text
[审计] 准备调用模型，messages=1
```

这是 `@before_model` 打出来的。它说明 Agent 准备调用模型，此时上下文里有 1 条消息。

后面你会看到：

```text
[审计] 准备调用模型，messages=3
[审计] 准备调用模型，messages=5
```

这不是重复执行错了，而是 Agent 的正常循环：模型先判断要查权限，工具执行后模型再继续判断，创建任务被拦截后模型还要再生成最终回复。每一轮模型调用前，都会经过 `before_model`。

**第二层：本地模型重试，再切云端**

```text
[模型调用] 本地 qwen2.5 第 1 次尝试
[模型调用] 本地模型失败：ConnectError
[模型调用] 本地 qwen2.5 第 2 次尝试
[模型调用] 本地模型失败：ConnectError
[模型调用] 本地 qwen2.5 第 3 次尝试
[模型调用] 本地模型失败：ConnectError
[模型调用] 本地模型连续失败，切换到云端 deepseek-v4-flash
```

这是 `@wrap_model_call` 在工作。

它包住了模型调用，所以能做到：先用本地 `qwen2.5:14b` 试 3 次；如果 Ollama 没开，出现 `ConnectError`；连续失败后，再把同一次请求切到云端 `deepseek-v4-flash`。

注意，这段日志可能出现多次。因为 Agent 不止调用一次模型，每一轮模型调用都会执行这套“本地重试 -> 云端兜底”的逻辑。

**第三层：工具调用审计和业务拦截**

```text
[工具审计] 准备调用 check_export_permission，args={'user_role': 'finance', 'region': 'east'}
[工具审计] check_export_permission 执行完成，耗时 0.5ms
```

这是 `@wrap_tool_call` 在记录工具调用。权限查询只是本地字典判断，所以耗时是毫秒级。

后面这段更关键：

```text
[工具审计] 准备调用 create_export_task，args={..., 'estimated_rows': 200000, ...}
```

模型确实尝试创建导出任务，但 `wrap_tool_call` 发现 `estimated_rows=200000`，超过了我们设置的 `100000` 上限，于是没有真正执行 `create_export_task`，而是返回了一条工具消息：需要人工审批或拆分导出范围。

所以最终回复里才会出现：

```text
预计导出行数超过 100,000，需要先走人工审批或拆分导出范围。
```

这就是自定义 Middleware 的价值：模型可以提出动作，但真正执行前，工程规则还能再把一道关。

这比单纯配置 `ModelRetryMiddleware` 或 `ModelFallbackMiddleware` 更贴近当前需求：**先重试同一个本地模型，还是不行，再换备用模型；工具动作也要按业务规则拦住。**

## 五、class 写法：把规则打包成一个 Middleware

Decorator 适合快速写小规则。

如果规则越来越多，或者需要配置参数，比如最大导出行数，就更适合写成 `AgentMiddleware` 子类。

In [ ]:
class ExportGovernanceMiddleware(AgentMiddleware):
    """订单报表导出 Agent 的治理规则。"""

    def __init__(self, max_rows: int = 100_000):
        self.max_rows = max_rows

    def before_model(self, state: AgentState, runtime):
        """模型调用前：记录上下文规模。"""
        print(f"[ExportGovernance] messages={len(state['messages'])}")
        return None

    def after_model(self, state: AgentState, runtime):
        """模型调用后：记录模型是否发起工具调用。"""
        last_message = state["messages"][-1]
        tool_calls = getattr(last_message, "tool_calls", [])
        print(f"[ExportGovernance] tool_calls={len(tool_calls)}")
        return None

    def wrap_tool_call(self, request, handler):
        """工具调用时：拦截超大导出任务。"""
        tool_name = request.tool_call["name"]
        args = request.tool_call.get("args", {})

        if tool_name == "create_export_task" and args.get("estimated_rows", 0) > self.max_rows:
            return ToolMessage(
                name=tool_name,
                tool_call_id=request.tool_call["id"],
                content=f"预计导出行数超过 {self.max_rows}，请先走人工审批或拆分导出范围。",
            )

        return handler(request)


class_style_agent = create_agent(
    model=local_model,
    tools=[check_export_permission, create_export_task],
    middleware=[ExportGovernanceMiddleware(max_rows=100_000)],
    system_prompt="你是订单报表导出 Agent。创建任务前先检查权限。",
)

class 写法的好处是：

- 多个 hook 可以放在一个类里
- 参数可以通过 `__init__` 传入
- 更适合团队复用和测试

简单规则用 decorator，复杂规则用 class。三行日志没必要硬写成类，代码不是简历。

## 六、小结：什么时候该自己写 Middleware

这篇的重点不是把所有 hook 背下来，而是知道它们该放在哪里。

| 需求 | 推荐做法 |
|---|---|
| 官方已经有内置 Middleware | 优先用内置 |
| 只是某个工具自己的业务逻辑 | 写在工具函数里 |
| 要拦截所有模型调用 | `@wrap_model_call` |
| 要拦截所有工具调用 | `@wrap_tool_call` |
| 要在模型前后记录审计信息 | `@before_model` / `@after_model` |
| 多个规则要复用 | `AgentMiddleware` 子类 |

几个容易踩坑的点：

- Middleware 顺序会影响结果。多个 middleware 都拦截模型或工具时，谁先执行、谁包住谁，会影响日志、重试和兜底行为。
- `handler(request)` 才是真正继续执行。如果在 `wrap_model_call` 或 `wrap_tool_call` 里不调用它，模型或工具就不会继续执行。
- `request.override(...)` 只改当前这一次请求，比如临时把模型换成 `cloud_model`，不是永久修改 Agent。
- 工具拦截时可以返回 `ToolMessage`，不一定要抛异常。让 Agent 知道“为什么没执行”，它才能给用户一个正常解释。
- 自定义 Middleware 不要滥用。所有模型或工具都要统一经过的规则，才适合放进 Middleware。

这就是 Middleware 最有意思的地方：它不是替 Agent 做决定，而是把 Agent 的每一步放进可控的工程流程里。

参考：

- LangChain Custom middleware：https://docs.langchain.com/oss/python/langchain/middleware/custom
- LangChain middleware reference：https://reference.langchain.com/python/langchain/agents/middleware
- LangChain wrap_model_call：https://reference.langchain.com/python/langchain/agents/middleware/types/wrap_model_call
- LangChain wrap_tool_call：https://reference.langchain.com/python/langchain/agents/middleware/types/wrap_tool_call